---
---
# RANDOMNESS AND PROBABILITY

---
---

## ROADMAP: where we've been, and where we're going

In the last lecture we took *many* simple random samples from the Titanic data and studied the **sampling distribution** of an estimate. We learned that:

* estimates from random samples tend to land *near* the true population value,
* larger samples tend to provide estimates that land *closer* to the truth, but
* we can always get an unlucky sample whose estimate is far off.

Consider the words in those statements: *"tend to"*, *"near"*, *"closer"*, *"unlucky"*, *"far off"*. These are vague terms, but data-driven decisions require something sharper; something more quantifiable. 

This is what we do with **probability**, which you can think of as a language for communicating about uncertainty using *numbers* instead of vague words.

We ended the last lecture with a promise:

> *"In practice, we only have one sample. So, what can we do in that case? The answer is to build proxy sampling distributions and calculate confidence intervals. But before we do that, we need to learn about **probability**."*

This is where we begin now.


---
---

## Thinking Probabilistically

### 🗣 ACTIVITY: How likely is *likely*?

**THINK.** In the code cell below, replace each `np.nan` with a number between 0 to 100 reflecting what you think each phrase communicates when you read it in a news story or an official report. Go with your gut. Assume 0 means impossible and 100 means certain.

**PAIR.** Find a neighbour. Type their numbers into the second dictionary, then run the comparison cell. Which phrase do the two of you disagree about the most?

**SHARE.** We'll poll the room on the phrase *"real possibility"*. What range of answers do we get? How would you feel if that phrase appeared in a flood warning? In a medical report about your surgery?

In [7]:
# THINK: replace each np.nan with your own number between 0 and 100
import numpy as np
my_numbers = {
    "almost certain":   np.nan,
    "very likely":      np.nan,
    "likely":           np.nan,
    "real possibility": np.nan,
    "unlikely":         np.nan,
    "very unlikely":    np.nan,
    "almost no chance": np.nan
}

In [6]:
# PAIR: replace each np.nan with your neighbour's numbers
neighbour_numbers = {
    "almost certain":   np.nan,
    "very likely":      np.nan,
    "likely":           np.nan,
    "real possibility": np.nan,
    "unlikely":         np.nan,
    "very unlikely":    np.nan,
    "almost no chance": np.nan
}

In [11]:
# Compare your answers. Which phrase has the biggest disagreement?
import pandas as pd
comparison = pd.DataFrame({"Phrase": list(my_numbers.keys()),
                           "Me": list(my_numbers.values()),
                           "My Neighbour": list(neighbour_numbers.values())})
comparison["Absolute Difference"] = (comparison["Me"] - comparison["My Neighbour"]).abs()
comparison.sort_values(by = "Absolute Difference", ascending = False)

,Phrase,Me,My Neighbour,Absolute Difference
0,almost certain,NaN,NaN,NaN
1,very likely,NaN,NaN,NaN
2,likely,NaN,NaN,NaN
3,real possibility,NaN,NaN,NaN
4,unlikely,NaN,NaN,NaN
5,very unlikely,NaN,NaN,NaN
6,almost no chance,NaN,NaN,NaN


### Probability: a number that measures uncertainty

A **probability** is a number between 0 and 1 (also expressed between 0% and 100% as percent **chance**) that **measures how likely an event is to occur**.

* A probability of 0 means the event is impossible.
* A probability of 1 means the event is certain.
* Probabilities between 0 and 1 measure degrees of *likelihood*. 
    * An event with probability close to 1 is more likely than an event with probability close to 0.
    * A probability of 0.5 expresses uncertainty in something like the toss of a fair coin: *heads* and *tails* are equally likely; they each have a 50% chance of happening.

We see statements involving probability all the time. 

* *There is an 80% chance of rain this afternoon.*
* *The odds of winning the lottery are 1 in 33 million.*
* *This common surgical procedure has a 98% success rate.*
* *The underdog team only has a 10% chance of making the playoffs this season.*

But what do these statements *actually mean*? That's what we're here to answer.

---
---

## The Language of Probability

To talk precisely about probability, we need some vocabulary.

### Random processes, outcomes, sample spaces, and events

* A **random process** is any process whose result we can't predict with certainty in advance, but whose possible results we *can list*. 
    * Important: random doesn't mean chaotic or mysterious, it just means *uncertain*. For example, a coin flip is random. We don't know what *will* happen but we do know what *can* happen.
* An **outcome** is one single, specific result of the random process.
* The **sample space** is the collection of all possible outcomes. 
    * Important: the term "sample" here is *completely distinct* from the term "sample" we've used to mean a subset of a population.
* An **event** is a collection of outcomes. 
    * We can also think of an event as a statement about the outcome that is either true or false once the process is complete.

| | Flip a coin | Roll a die | Pick one Titanic passenger at random |
|---|---|---|---|
| **Random process** | flip the coin | roll the die | `titanic.sample(n=1)` |
| **An outcome** | heads | a 4 | passenger number 1017 |
| **Sample space**| {heads, tails} | {1, 2, 3, 4, 5, 6} | all 2207 passengers and crew |
| **An event** | "the coin lands heads" | "the die shows an even number"| "the passenger survived" |

Python can simulate each of these random processes. **Run each of the cells below several times** and watch the results change:

**A coin flip:**

In [ ]:
np.random.choice(["heads", "tails"])

**A die roll:**

In [ ]:
np.random.choice([1, 2, 3, 4, 5, 6])

**One passenger picked at random from the Titanic data:**

In [63]:
titanic = pd.read_csv("https://raw.githubusercontent.com/ntstevens/DATSC101/refs/heads/main/data/titanic_passenger_info.csv")

In [ ]:
titanic.sample(n = 1)

Notice how we can't predict what any one run will give, but we *do* know all the things that could happen. That's what makes these random processes.

### Events are logical statements 

Previously, we filtered rows of a data frame using *logical statements*, like 

```python 
mlb_salaries["Team"] == "TOR"
```

We can use exactly the same idea to define **events** for the process "pick one passenger at random":

* an event is a True/False statement about the passenger,
* the passengers for whom the statement is `True` are the outcomes that compose the event.

Let's define four events about Titanic passengers, and collect them in one data frame so each *column* is an event:

In [102]:
survived    = (titanic["survived"] == "yes")
female      = (titanic["gender"] == "female")
child       = (titanic["age"] < 18)       
first_class = (titanic["class"] == "1st")

events = pd.DataFrame({"survived": survived,
                       "female": female,
                       "child": child,
                       "first_class": first_class})
events.head(10)

,survived,female,child,first_class
0,False,False,False,False
1,False,False,True,False
2,False,False,True,False
3,True,True,False,False
4,True,True,True,False
5,True,False,False,False
6,False,False,False,False
7,True,True,False,False
8,True,False,False,False
9,True,False,False,False


How many outcomes (passengers) are in each event? Summing counts the `True` values.

In [103]:
events.sum()

survived       711
female         489
child          200
first_class    324
dtype: int64

Combining events works just like combining logical statements. Given two events **A** and **B**, here's how we map between plain English and Python:

* **A and B** (both events happen) is written `A & B`
* **A or B** (at least one of the events happens, possibly both) is written `A | B`
* **not A** (event A does *not* happen) is written `~A`
    * This one is new: the "tilde" `~` flips every `True` to `False` and vice versa.

In [104]:
# Passengers who were female AND survived
(female & survived).sum()

359

In [105]:
# Passengers who were female OR survived (or both)
(female | survived).sum()

841

In [106]:
# Passengers who did NOT survive
(~survived).sum()

1496

* Events **A** and **B** are **mutually exclusive** if they *cannot both happen* at the same time.
    * In other words: no outcome belongs to both events. For instance, a passenger can't be in both first class and second class:

In [101]:
second_class = (titanic["class"] == "2nd")
(first_class & second_class).sum()

0

Zero passengers are in both events, so "first class" and "second class" are mutually exclusive. On the other hand, "female" and "survived" are *not* mutually exclusive, since we found plenty of passengers who are both.

### Example: The sample space of two coins
Suppose we flip a coin *twice*. What's the sample space? Each outcome is a *pair* of results. A **nested `for` loop** (a loop inside a loop) can list them all: the inner loop runs *completely* for every step of the outer loop.

In [107]:
two_flips = []
for first_flip in ["H", "T"]:
    for second_flip in ["H", "T"]:
        two_flips.append(first_flip + second_flip)

two_flips

['HH', 'HT', 'TH', 'TT']

The sample space has 4 outcomes: `HH`, `HT`, `TH`, `TT`. The event "*at least one head*" is the collection {`HH`, `HT`, `TH`}, which contains 3 of the 4 outcomes.

Now it's your turn to build a sample space.

### 💻 ACTIVITY: The sample space of two dice

Roll two dice, one red and one green. Each outcome is a pair (red number, green number).

1. Complete the nested loop to build the sample space, and store it as a data frame `dice` with one row per outcome. How many outcomes are there?
2. Define three events as boolean columns, and count the outcomes in each:
    * `total_is_7`: the two dice add up to 7
    * `doubles`: both dice show the same number
    * `has_a_6`: at least one die shows a 6
3. Which pairs of these three events are mutually exclusive? Check using `&`.

In [ ]:
# 1. Build the sample space (every (red, green) pair). How many outcomes are there?
sample_space = []
for red in range(1, 7):
    for green in range(1, 7):                    
        sample_space.append((red, green))
dice = pd.DataFrame(sample_space, columns = ["red", "green"])
print(dice)

In [ ]:
# 2. Define the three events, then count how many outcomes are in each
total_is_7 = ____          # the two dice add up to 7
doubles    = ____          # both dice show the same number
has_a_6    = ____          # at least one die shows a 6
print(total_is_7.sum(), doubles.sum(), has_a_6.sum())

In [ ]:
# 3. Which pairs of events are mutually exclusive?
# Your code here

<details>
<summary><b>Show Solution</b></summary>

```python
# 1. How many outcomes?
print(len(dice))      

# 2. Define the events and count
total_is_7 = ((dice["red"] + dice["green"]) == 7)
doubles    = (dice["red"] == dice["green"])
has_a_6    = (dice["red"] == 6) | (dice["green"] == 6)
print(total_is_7.sum(), doubles.sum(), has_a_6.sum()) 

# 3. Overlaps
print((total_is_7 & doubles).sum())   
print((total_is_7 & has_a_6).sum())   
print((doubles & has_a_6).sum())     

```

The only mutually exclusive pair is `total_is_7` and `doubles`: doubles always add up to an even number, so they can never add up to 7.

</details>

We now have *counts* of outcomes in each event. Next we learn how to turn counts into probabilities, and the rules probabilities must follow.